# Statistics for Data Scientists

Notebook 05 taught you to *look* at data. This notebook teaches you to *make claims* about what you see.

A bar in a chart is taller than another bar. So what? Statistics tells you whether the difference is real or could plausibly be noise. Two columns trend together in a scatterplot. So what? Statistics tells you how strong that relationship is and whether it's reproducible.

Four topics, the smallest useful set.

1. **Descriptive statistics** — summarizing a column in a few numbers.
2. **Probability distributions** — the shapes data takes, and how to sample them.
3. **Hypothesis testing** — the framework for deciding whether a difference is real.
4. **Correlation** — measuring how two variables move together.

We use `scipy.stats` for the formal tests. The PyData stack has a heavier-duty stats library too — `statsmodels` — but for the daily questions a data scientist asks, `scipy.stats` is enough.


## Descriptive statistics

A column's distribution can be summarized by a handful of numbers.

| Metric | What it tells you |
|---|---|
| `mean` | The arithmetic average. Pulled by outliers. |
| `median` | The middle value. Robust to outliers. |
| `std` | Standard deviation — how spread out the values are around the mean. |
| `quantile(0.25)`, `quantile(0.75)` | First and third quartiles. The interquartile range (IQR) is `Q3 − Q1`. |
| `min`, `max` | The endpoints. Often hide outliers. |
| `skew` | Asymmetry. Positive = long right tail (income); negative = long left tail. |
| `kurtosis` | "Tailedness" — high kurtosis means lots of extreme values. |

The mental model: mean and median tell you the *center*; std and IQR tell you the *spread*; skew and kurtosis tell you the *shape*.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# Realistic monthly income — log-normal, right-skewed
income = pd.Series(rng.lognormal(mean=11, sigma=0.6, size=500).astype(int))

print("count:    ", income.count())
print("mean:     ", round(income.mean()))
print("median:   ", income.median())
print("std:      ", round(income.std()))
print("Q1:       ", income.quantile(0.25))
print("Q3:       ", income.quantile(0.75))
print("IQR:      ", income.quantile(0.75) - income.quantile(0.25))
print("min/max:  ", income.min(), "/", income.max())
print("skew:     ", round(income.skew(), 2))
print("kurtosis: ", round(income.kurtosis(), 2))


## Mean vs median — pick the right summary

For symmetric distributions, mean and median agree. For skewed distributions, they don't — and the choice matters.

**Income** is the classic case. Most customers earn modestly; a few earn an order of magnitude more. The mean is pulled toward those high earners. The median sits in the middle of the actual population.

When you report a "typical" value, the **median** is almost always more honest for skewed data. Mean is appropriate for symmetric data, or when you need a quantity that's mathematically tractable for downstream calculations.

The same logic applies to spread: standard deviation assumes a symmetric distribution; the IQR is the median's robust cousin.


In [ ]:
# Normal-looking customer balances
balances = pd.Series([45_000, 52_000, 38_000, 61_000, 47_000, 55_000, 49_000, 41_000, 58_000, 50_000])
print("without outlier — mean:", round(balances.mean()), " median:", balances.median())

# One billionaire customer joins the bank
with_outlier = pd.concat([balances, pd.Series([50_000_000])], ignore_index=True)
print("with    outlier — mean:", round(with_outlier.mean()), "median:", with_outlier.median())

# The mean roughly 100xs. The median barely moves.


## Probability distributions

A **distribution** is a function that describes how likely each value is. The three you will meet most in financial data.

| Distribution | Shape | Examples in Fintech Bank |
|---|---|---|
| **Normal** (Gaussian) | Bell curve, symmetric | Daily returns, measurement noise |
| **Uniform** | Flat — every value equally likely | Random IDs, A/B test assignment |
| **Lognormal** | Right-skewed; `log(x)` is normal | Income, account balances, loan amounts, transaction sizes |

NumPy's `default_rng().normal(...)`, `.uniform(...)`, `.lognormal(...)` sample from each. `scipy.stats` gives you the full distribution object — PDF, CDF, percentile-point function, fitting — when you need more than just sampling.


In [ ]:
normal = rng.normal(loc=50, scale=10, size=2000)
uniform = rng.uniform(low=0, high=100, size=2000)
lognormal = rng.lognormal(mean=3, sigma=0.6, size=2000)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
sns.histplot(normal,    bins=30, ax=axes[0]); axes[0].set_title("Normal")
sns.histplot(uniform,   bins=30, ax=axes[1]); axes[1].set_title("Uniform")
sns.histplot(lognormal, bins=30, ax=axes[2]); axes[2].set_title("Lognormal — right-skewed")
fig.tight_layout()
plt.show()


## Why distributions matter — the log transform

Most ML and statistical methods assume the input is roughly **normal**. They behave badly on skewed inputs — outliers dominate, regression coefficients explode, distance metrics get distorted.

The cure for right-skewed data is the **log transform**. If `X` is lognormal, then `log(X)` is normal by construction. This single transformation makes income, balances, and transaction amounts behave well in any downstream model.

You will see this pattern over and over in notebook 07 feature engineering: identify the skewed columns, apply `np.log1p` (which is `log(1 + x)` and handles zeros gracefully), then proceed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

sns.histplot(income, bins=40, ax=axes[0])
axes[0].set_title(f"Original income — skew = {round(income.skew(), 2)}")

log_income = np.log(income)
sns.histplot(log_income, bins=40, ax=axes[1])
axes[1].set_title(f"log(income) — skew = {round(log_income.skew(), 2)}")

fig.tight_layout()
plt.show()


## Hypothesis testing — the framework

The framework for deciding whether a difference is real or just noise.

1. State a **null hypothesis (H₀)** — the boring claim, usually "there is no difference."
2. State an **alternative hypothesis (H₁)** — the interesting claim, "there is a difference."
3. Compute a **test statistic** from the data.
4. Compute the **p-value** — the probability of seeing data this extreme *if H₀ were true*.
5. If `p < α` (conventionally `α = 0.05`), reject H₀.

A p-value answers exactly one question: *"if there really were no effect, how surprising would my data be?"* Small p means surprising means the no-effect assumption probably does not hold.

Two tests cover the bulk of daily work — `ttest_ind` for comparing two numeric groups, and `chi2_contingency` for comparing categorical breakdowns.


## Two-sample t-test — comparing two means

**Question:** do retail and wealth-segment customers earn different mean incomes?

- H₀ — the two segments have the same mean income.
- H₁ — the two segments have different mean incomes.

`scipy.stats.ttest_ind` does the work. It returns a test statistic and a p-value. If you suspect the two groups have different variances — almost always true for income — pass `equal_var=False`. That is **Welch's t-test**, the safe default.

Always pair the p-value with an **effect size**. Cohen's `d` is the standard: the difference in means divided by the pooled standard deviation. `d ≈ 0.2` is small, `0.5` is medium, `0.8+` is large.


In [ ]:
retail_income = rng.lognormal(mean=10.8, sigma=0.5, size=300)
wealth_income = rng.lognormal(mean=11.6, sigma=0.5, size=80)

t_stat, p_value = stats.ttest_ind(retail_income, wealth_income, equal_var=False)
print(f"retail mean:  {retail_income.mean():,.0f}")
print(f"wealth mean:  {wealth_income.mean():,.0f}")
print(f"t-statistic:  {t_stat:.3f}")
print(f"p-value:      {p_value:.2e}")
print("reject H0?    ", p_value < 0.05)

# Effect size — how big is the difference, in standard deviations?
pooled_std = np.sqrt((retail_income.var() + wealth_income.var()) / 2)
cohens_d = (wealth_income.mean() - retail_income.mean()) / pooled_std
print(f"Cohen's d:    {cohens_d:.2f}  (>0.8 = large effect)")


## Chi-square test — comparing categorical distributions

**Question:** is customer segment independent of city? Or do some cities have a higher proportion of wealth-segment customers than others?

- H₀ — segment and city are independent.
- H₁ — they are not; the segment distribution differs across cities.

The test takes a **contingency table** (a cross-tab of counts) and asks whether the observed counts deviate meaningfully from what you would expect under independence. `pd.crosstab(...)` builds the table; `stats.chi2_contingency(table)` runs the test.


In [ ]:
n = 400
cities = rng.choice(["Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Pune"], n)

# Bias: Bengaluru and Mumbai have proportionally more wealth-segment customers
wealth_probs = {"Bengaluru": 0.35, "Mumbai": 0.30, "Hyderabad": 0.15, "Chennai": 0.10, "Pune": 0.12}
segments = np.array(["wealth" if rng.random() < wealth_probs[c] else "retail" for c in cities])

customers = pd.DataFrame({"city": cities, "segment": segments})
table = pd.crosstab(customers["city"], customers["segment"])
print("contingency table:")
print(table)

chi2, p_value, dof, expected = stats.chi2_contingency(table)
print(f"\nchi-square stat:    {chi2:.3f}")
print(f"p-value:            {p_value:.2e}")
print(f"degrees of freedom: {dof}")
print("reject H0?         ", p_value < 0.05)


## Correlation — Pearson vs Spearman

Two columns move together. How tightly? And in what *kind* of relationship?

| Measure | What it captures | Use when |
|---|---|---|
| **Pearson** | **Linear** correlation. Sensitive to outliers and to non-linear shapes. | Both columns are continuous and roughly linearly related. |
| **Spearman** | **Monotonic** correlation. Computed on ranks, so robust to non-linearity and outliers. | You suspect a non-linear but consistent trend, or your data has heavy tails. |

Both return a coefficient in `[-1, +1]` and a p-value testing the null *"no correlation."* When in doubt, look at both. A large gap between them is a signal that the relationship is monotonic but not linear — go look at the scatterplot.


In [ ]:
x = rng.uniform(1, 100, 200)
y_linear = 2 * x + rng.normal(0, 20, 200)
y_curve  = np.log(x) * 10 + rng.normal(0, 1, 200)   # monotonic but not linear

# Linear relationship — pearson and spearman agree
p_r, p_p = stats.pearsonr(x, y_linear)
s_r, s_p = stats.spearmanr(x, y_linear)
print(f"linear  — pearson:  r={p_r:.3f}, p={p_p:.2e}")
print(f"          spearman: r={s_r:.3f}, p={s_p:.2e}")

# Monotonic non-linear — spearman holds; pearson drops
p_r, p_p = stats.pearsonr(x, y_curve)
s_r, s_p = stats.spearmanr(x, y_curve)
print(f"\ncurve   — pearson:  r={p_r:.3f}, p={p_p:.2e}")
print(f"          spearman: r={s_r:.3f}, p={s_p:.2e}")
print("(gap between the two → monotonic but curved — plot it before you trust pearson)")


## A note — p-values aren't enough

Two traps that ruin most "statistically significant" claims in business decks.

**Effect size matters more than significance.** With a large enough sample, *any* difference becomes statistically significant. A p-value of 1e-9 with a Cohen's d of 0.05 means *"yes, there is a difference, but it is so small that no one cares."* Always report or check the effect size alongside the p-value.

**Multiple testing inflates false positives.** If you run 20 t-tests at α = 0.05, you expect roughly 1 false positive even when nothing is real. Either correct for it (`statsmodels.stats.multitest.multipletests` with `method="bonferroni"` or `"fdr_bh"`) or pre-register the single test you actually care about before looking at the data.

These two failures are where most "we have a significant result" claims fall apart under scrutiny.


## What's next

You can now describe a column, identify its distribution, test whether two groups differ, and measure how two columns relate. That is the formal language for everything notebook 05's plots were showing you visually.

Notebook 07 brings the full track together in an end-to-end **EDA and feature engineering** pass on Fintech Bank data — descriptive stats, distribution checks, missing-data audits, correlation pruning, log transforms, and the categorical encodings that hand a clean DataFrame off to `machine-learning/`.

Two from-memory exercises.

1. Build a `card_transactions` Series of 500 amounts using `lognormal`. Compute its skew, then apply `np.log` and recompute. By how much did the skew drop?
2. Build two samples of card transactions — one labeled `"groceries"`, one labeled `"electronics"` — drawn from lognormals with different means. Run Welch's t-test and compute Cohen's `d`. Is the difference statistically significant? Is it practically significant?
